# 01. Team Formation

A solo agent is a function. A *team* is an organization. The moment you have more than one agent collaborating on a goal you need answers to questions a single agent never has to ask: *who is on this team, what role do they play, who is allowed to speak to whom, who do I trust?* `arcteam` is the package that owns those answers.

This notebook is the first stop for `arcteam`. We'll cover how a team is configured, how members are registered with cryptographic identity, how role-based addressing works, and how a minimal team comes together end-to-end. Everything runs in-process against `MemoryBackend` — no filesystem, no network, no API key.

**You will learn:**

- What `TeamConfig` carries and the defaults you usually leave alone
- The `Entity` data model — agents and users, with roles and capabilities
- How `EntityRegistry` registers, looks up, filters, and removes members
- How DIDs from `arctrust` bind a registered entity to a verifiable identity
- How to assemble a minimal team end-to-end with `MemoryBackend` + `AuditLogger`
- How team membership produces tamper-evident audit events
- The lifecycle of a team: formation, operation, dissolution
- Where `arcteam` sits in the four-pillar story (and where it stops)

If you haven't read `arctrust/01-identity-did.ipynb` yet, do that first. We cross-reference it here whenever DIDs come up — this notebook does not re-derive DID basics.

## 1. Setup

The setup cell below is the standard Arc walkthrough preamble — it locates the repo root, adds every package's `src/` (and `tests/` where present) to `sys.path`, and loads any `.env` at the root. Identical across every Arc notebook so you can skim past once you've seen it.

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Smoke-import the public surface we'll touch in this notebook. If any of these fail, fix the environment before going further — every subsequent cell assumes them.

In [2]:
import hashlib

import arcteam
from arctrust.signer import ED25519, InProcessSigner
from arcteam import (
    AuditLogger,
    Entity,
    EntityRegistry,
    EntityType,
    MemoryBackend,
    TeamConfig,
)

print(f"arcteam version: {arcteam.__version__}")
print(
    "Loaded:",
    [
        AuditLogger.__name__,
        Entity.__name__,
        EntityRegistry.__name__,
        EntityType.__name__,
        MemoryBackend.__name__,
        TeamConfig.__name__,
    ],
)

arcteam version: 0.5.0
Loaded: ['AuditLogger', 'Entity', 'EntityRegistry', 'EntityType', 'MemoryBackend', 'TeamConfig']


## 2. Why teams — multi-agent coordination beyond solo agents

Arc's stack splits responsibilities cleanly. A single agent (`arcagent`) is a self-contained nucleus — identity, tool registry, policy pipeline, module bus. The runtime loop (`arcrun`) drives a single agent's reason-act cycle. The LLM call surface (`arcllm`) is provider-agnostic. None of those packages know what to do when *more than one* agent needs to coordinate on a shared goal — and that's deliberate. Stacking concerns onto the agent nucleus is how you get unmaintainable Frankenstein agents.

`arcteam` owns the multi-agent layer. From `packages/arcteam/CLAUDE.md`:

| Concern | arcteam's job | NOT arcteam's job |
|---|---|---|
| Team formation | Create, join, leave, dissolve teams | Individual agent lifecycle |
| Task distribution | Assign, delegate, track work across agents | Execute tasks (that's arcrun) |
| Inter-agent comms | Route messages, signed channels, broadcast | LLM calls (that's arcllm) |
| Roster management | Membership, roles, capability discovery | Agent identity (that's arctrust) |
| Team lifecycle | Formation → Active → Dissolving → Dissolved | Agent config (arcagent) |
| Team telemetry | Team-level audit events | Per-agent telemetry (arcagent) |

The mental model: **`arcteam` orchestrates `ArcAgent` instances** — it does not replace or duplicate agent internals. A team is a *registry* of identities plus the conventions that let those identities address one another, exchange signed messages, and leave a tamper-evident trail.

This notebook covers the first half of that — formation and roster. Channels, messaging, persistence, and consensus get their own notebooks (`02`–`04`).

## 3. `TeamConfig` — what it carries

`TeamConfig` is a small Pydantic model that pins down **where the team's state lives** and **how big a single message can be**. It's deliberately tiny — most teams take the defaults and never look at it again. The defaults align with `arcteam`'s "secure by default" posture (an Ed25519 signer supplied by the caller, 64 KB body cap to bound memory and audit size).

| Field | Default | Meaning |
|---|---|---|
| `root` | `~/.arc/team` | Shared-file directory root used by `TeamFileStore` (unrelated to message/audit storage) |
| `max_body_bytes` | `65536` (64 KB) | Hard cap on a single `Message.body` |
| `default_poll_limit` | `10` | Default page size for poll/read APIs |

Note what's *not* there — no team name, no member list, no policy. `TeamConfig` carries deployment-level knobs; team identity and roster live in the registry. This separation is intentional: a process can run many teams against one `TeamConfig`.

In [3]:
# Defaults out of the box.
default_cfg = TeamConfig()
print("root:                ", default_cfg.root)
print("max_body_bytes:      ", default_cfg.max_body_bytes)
print("default_poll_limit:  ", default_cfg.default_poll_limit)

root:                 /Users/joshschultz/.arc/team
max_body_bytes:       65536
default_poll_limit:   10


You override only what you need. Common overrides: pointing `root` at a project-local directory in tests, or shrinking `max_body_bytes` for a deployment that wants smaller envelopes.

In [4]:
import tempfile

# Project-local team root for an integration test or sandbox deployment.
sandbox_root = Path(tempfile.mkdtemp(prefix="arc_team_demo_"))
test_cfg = TeamConfig(root=sandbox_root, max_body_bytes=8192, default_poll_limit=25)
print("sandbox root:    ", test_cfg.root)
print("max_body_bytes:  ", test_cfg.max_body_bytes)
print("poll_limit:      ", test_cfg.default_poll_limit)
# Untouched fields keep defaults.
assert test_cfg.default_poll_limit == 25
print("default_poll_limit (custom):", test_cfg.default_poll_limit)

sandbox root:     /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_team_demo_1db6deji
max_body_bytes:   8192
poll_limit:       25
default_poll_limit (custom): 25


`TeamConfig` is a Pydantic v2 model, so you get validation, `.model_dump()`, and `.model_validate()` for free. Round-trip it through JSON the way you'd serialize any other Arc config.

In [5]:
raw = test_cfg.model_dump(mode="json")
print(raw)

restored = TeamConfig.model_validate(raw)
assert restored.max_body_bytes == 8192
print("Round-trip OK:", restored.max_body_bytes == test_cfg.max_body_bytes)

{'root': '/var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_team_demo_1db6deji', 'max_body_bytes': 8192, 'default_poll_limit': 25}
Round-trip OK: True


## 4. `Entity` and `EntityRegistry` — the team roster

A *team* is a collection of `Entity` records held in an `EntityRegistry`. An `Entity` is the smallest unit of membership — a Pydantic model that names *who* is on the team and *what they can do*. The registry handles the CRUD surface plus role-based queries.

### `Entity` shape

| Field | Type | Meaning |
|---|---|---|
| `id` | `str` | URI-form identifier — `agent://name` or `user://name` |
| `did` | `str` | **Required.** The entity's cryptographic identity from `arctrust` (arcteam never mints identities — the caller supplies this). No entity exists without a DID. |
| `handle` | `str` | **Required.** Unique display name for `@mention` / URI addressing; uniqueness enforced at registration. |
| `name` | `str` | Human-readable label |
| `type` | `EntityType` | `AGENT` or `USER` |
| `roles` | `list[str]` | Tags used for role-based addressing |
| `capabilities` | `list[str]` | Free-form capability tags (`"summarize"`, `"sql"`, …) |
| `created` | `str` | ISO-8601 timestamp; auto-set on register if blank |
| `status` | `EntityStatus` | Registration state. **Today the enum defines exactly one member — `active`.** The field is typed for future lifecycle states, but nothing else is valid yet — see the gotcha in section 6. |
| `workspace_path` | `str \| None` | SPEC-019 — agent's workspace dir; `None` for users |

Entities are addressed by typed URIs (`agent://`, `user://`). The registry rejects duplicates by ID — you cannot register the same identity twice.

`did` and `handle` are both fail-closed required fields (not the loose `capabilities`-list convention older docs describe) — every `Entity` is DID-keyed from construction.

In [6]:
from arctrust import AgentIdentity

# Side table: entity id -> the AgentIdentity that minted its `did`, so later
# cells can retrieve the private key behind an already-registered Entity
# without generating a mismatched second identity.
_entity_identities: dict[str, AgentIdentity] = {}


def _entity(*, id: str, name: str, type: EntityType, **kwargs) -> Entity:
    """Demo helper: `Entity` requires a real `did` + a `handle` (the URI's
    local part). Mint a throwaway identity so every call site below doesn't
    have to repeat the AgentIdentity boilerplate."""
    handle = id.split("://", 1)[1]
    identity = AgentIdentity.generate(org="demo", agent_type=handle)
    _entity_identities[id] = identity
    return Entity(id=id, name=name, type=type, did=identity.did, handle=handle, **kwargs)


async def _mutate_roles(registry: EntityRegistry, entity_id: str, roles: list[str]) -> Entity:
    """Read an entity, replace its `roles` list, write back via update().

    There is no `update_status()` — `update()` is the sole write path for an
    existing record. We mutate `roles` here (not `status`) because
    `EntityStatus` currently defines only `active`; role-tag mutation is the
    supported way to represent lifecycle changes today (see section 6)."""
    entity = await registry.get(entity_id)
    assert entity is not None
    entity.roles = roles
    await registry.update(entity)
    return entity


In [7]:
# Build one of each kind. Roles and capabilities are free-form lists.
planner = _entity(
    id="agent://planner",
    name="Planner",
    type=EntityType.AGENT,
    roles=["coordinator", "decomposer"],
    capabilities=["task.split", "task.assign"],
)
researcher = _entity(
    id="agent://researcher",
    name="Researcher",
    type=EntityType.AGENT,
    roles=["worker"],
    capabilities=["web.search", "doc.summarize"],
)
operator = _entity(
    id="user://josh",
    name="Josh",
    type=EntityType.USER,
    roles=["operator"],
)

for ent in (planner, researcher, operator):
    print(f"{ent.id:<22}  type={ent.type.value:<6}  roles={ent.roles}")

agent://planner         type=agent   roles=['coordinator', 'decomposer']
agent://researcher      type=agent   roles=['worker']
user://josh             type=user    roles=['operator']


### `EntityRegistry` API

`EntityRegistry` wraps a `StorageBackend` (we'll use `MemoryBackend` here) plus an `AuditLogger`. It exposes a small async surface:

| Method | Purpose |
|---|---|
| `register(entity)` | Insert a new entity; raises if the ID already exists |
| `get(entity_id)` | Read one entity by ID; returns `None` if missing |
| `list_entities(role=None)` | Enumerate all entities, optionally filtered |
| `by_role(role)` | Sugar for `list_entities(role=...)` — used for role-based addressing |
| `update(entity)` | Replace a record (role changes, workspace backfill, anything else) — the sole write path besides `register`; emits `entity.updated`. There is no separate `update_status`, and `status` itself is not a usable lifecycle knob today (see section 6) — mutate the field you actually need (e.g. `roles`) locally and call `update()`. |

Every mutation emits an audit event. We'll inspect the audit stream after we register a few entities.

In [8]:
# Build a registry on top of MemoryBackend + AuditLogger.
backend = MemoryBackend()
audit = AuditLogger(backend, signer=InProcessSigner(hashlib.sha256(b"demo-key-do-not-use-in-prod").digest(), ED25519))
await audit.initialize()

registry = EntityRegistry(backend, audit)
print("Registry ready:", type(registry).__name__)
print("Backed by:    ", type(backend).__name__)

Registry ready: EntityRegistry
Backed by:     MemoryBackend


Register all three entities. `register()` is a write-through that:

1. Rejects duplicates (raises `ValueError`).
2. Auto-sets `created` if blank.
3. Persists the record to the backend.
4. Emits an `entity.registered` audit event.

In [9]:
await registry.register(planner)
await registry.register(researcher)
await registry.register(operator)

# Read back to confirm everything stuck.
back = await registry.get("agent://planner")
assert back is not None
print("Re-read planner:")
print("  id:       ", back.id)
print("  name:     ", back.name)
print("  roles:    ", back.roles)
print("  created:  ", back.created)
print("  status:   ", back.status)

Re-read planner:
  id:        agent://planner
  name:      Planner
  roles:     ['coordinator', 'decomposer']
  created:   2026-07-09T12:08:30.514552+00:00
  status:    active


**Duplicate IDs are rejected.** This is a registry invariant: two members cannot share an ID.

In [10]:
# this raises — re-registering the same ID is a hard error
try:
    await registry.register(planner)
except ValueError as e:
    print("ValueError:", e)

ValueError: Entity already registered: did:arc:demo:planner/081db836


### Listing and filtering

`list_entities()` returns every entity. `by_role()` filters down to a single role tag. The role check is a simple membership test against `entity.roles`, so an entity with multiple roles surfaces in multiple role queries.

In [11]:
everyone = await registry.list_entities()
print(f"All entities ({len(everyone)}):")
for e in everyone:
    print(f"  {e.id:<22}  roles={e.roles}")

print()
workers = await registry.by_role("worker")
print(f"Workers ({len(workers)}):", [e.id for e in workers])

operators = await registry.by_role("operator")
print(f"Operators ({len(operators)}):", [e.id for e in operators])

# A role nobody has -> empty list, not an error.
ghosts = await registry.by_role("phantom")
print("Phantom role (empty):", ghosts)

All entities (3):
  user://josh             roles=['operator']
  agent://planner         roles=['coordinator', 'decomposer']
  agent://researcher      roles=['worker']

Workers (1): ['agent://researcher']
Operators (1): ['user://josh']
Phantom role (empty): []


### Updating a record — and a real gotcha with `status`

A team's roster isn't static — members change workspace, gain or lose role tags, or otherwise need their record refreshed. `update()` is the *only* write path for an existing record (there is no separate `update_status` method) — read, mutate the field you need, write back. Every call emits an `entity.updated` audit event.

One field to be careful with: `Entity.status` is typed `EntityStatus`, a `StrEnum` that **today defines exactly one member — `active`**. There is no `inactive` / `dissolved` / `offline` value to switch to. Because Pydantic v2 does not validate on plain attribute assignment by default, `entity.status = "inactive"` *succeeds silently* in memory — but the very next read goes through `Entity.model_validate(...)` (every `get()`/`list_entities()` call does this), which *does* enforce the enum, and that raises. We demonstrate the mutation that actually works (`workspace_path` backfill) here, then the status gotcha as a deliberate error case on a throwaway registry — the same way earlier cells showed `# this raises` for malformed DIDs.

In [12]:
# A legitimate update: backfill a workspace_path via update() (SPEC-019 path).
researcher_now = await registry.get("agent://researcher")
assert researcher_now is not None
researcher_now.workspace_path = str(sandbox_root / "agents" / "researcher")
await registry.update(researcher_now)

readback = await registry.get("agent://researcher")
assert readback is not None
print("researcher.workspace_path:", readback.workspace_path)
print("roles preserved:           ", readback.roles)
print("status (unchanged):        ", readback.status)

researcher.workspace_path: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_team_demo_1db6deji/agents/researcher
roles preserved:            ['worker']
status (unchanged):         active


In [13]:
# this raises — eventually. `entity.status = "inactive"` is accepted at
# assignment time (Entity has no `validate_assignment=True`) because
# `EntityStatus` currently defines only `active`. The corruption doesn't
# surface at the point of assignment — it surfaces on the next *validated*
# read. Run this on a throwaway registry so it can't leak into the rest of
# the notebook's shared state.
throwaway_backend = MemoryBackend()
throwaway_audit = AuditLogger(
    throwaway_backend,
    signer=InProcessSigner(hashlib.sha256(b"throwaway-key").digest(), ED25519),
)
await throwaway_audit.initialize()
throwaway_registry = EntityRegistry(throwaway_backend, throwaway_audit)

victim = _entity(id="agent://victim", name="Victim", type=EntityType.AGENT)
await throwaway_registry.register(victim)

broken = await throwaway_registry.get("agent://victim")
assert broken is not None
broken.status = "inactive"  # type: ignore[assignment]  — accepted in memory...
await throwaway_registry.update(broken)  # ...and persisted as a plain string

try:
    await throwaway_registry.get("agent://victim")  # ...rejected on the next validated read
except Exception as e:
    print(f"{type(e).__name__} on next validated read: status must be 'active'")

ValidationError on next validated read: status must be 'active'


/Users/joshschultz/Projects/arc/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='status', input_value='inactive', input_type=str])
  return self.__pydantic_serializer__.to_python(


**Removal — what `arcteam` does and doesn't expose.** The current registry surface intentionally has no `remove()` or `delete()` method on `EntityRegistry`. And — as the cell above just showed — `status` can't be repurposed as a lifecycle flag today, because `EntityStatus` defines only `active`. The practical, currently-supported way to take a member "out of rotation" is a role-tag convention: drop the member's functional roles (so `by_role()` queries stop returning them) and, if useful, add a marker role like `"retired"` or `"dissolved"`. The record stays in place — nothing is deleted — which is what actually preserves the audit trail; the mechanism is roles, not status, until `EntityStatus` grows more members. The underlying `StorageBackend.delete` exists for low-level callers, but day-to-day team operations mutate `roles` and call `update()` instead. This is by design: federal-tier audit requirements (NIST 800-53 AU-9) want immutable history.

In [14]:
# Soft-removal pattern: role-tag mutation rather than physical delete
# (status can't do this today — see the gotcha above).
researcher_now = await registry.get("agent://researcher")
assert researcher_now is not None
researcher_now.roles = [r for r in researcher_now.roles if r != "worker"] + ["retired"]
await registry.update(researcher_now)

final = await registry.get("agent://researcher")
assert final is not None
print(f"researcher roles after soft-remove: {final.roles}")
# Record still exists in the registry — audit chain is intact.
still_listed = await registry.list_entities()
print(f"Total entities after soft-remove: {len(still_listed)}")
workers_after = await registry.by_role("worker")
print(f"'worker' role query no longer includes researcher: {[e.id for e in workers_after]}")

researcher roles after soft-remove: ['retired']
Total entities after soft-remove: 3
'worker' role query no longer includes researcher: []


### Every mutation leaves an audit footprint

Each `register` and `update` (status changes included — there is no separate `update_status`) emits an Ed25519-signed audit record into `audit/audit`. The chain is verifiable end-to-end with `AuditLogger.verify_chain()` — same primitive `arctrust.audit` uses, just wrapped for team-scoped events.

In [15]:
# Read the audit stream directly off the backend.
records = await backend.read_stream("audit", "audit", after_seq=0)
print(f"Audit records: {len(records)}")
for r in records:
    print(f"  seq={r['audit_seq']}  event={r['event_type']:<24}  actor={r['actor_id']}")

# Verify the signed chain from end to end.
ok, last = await audit.verify_chain()
print()
print(f"chain valid? {ok}  (verified through seq={last})")
assert ok

Audit records: 5
  seq=1  event=entity.registered         actor=did:arc:demo:planner/081db836
  seq=2  event=entity.registered         actor=did:arc:demo:researcher/c3360492
  seq=3  event=entity.registered         actor=did:arc:demo:josh/bedf4967
  seq=4  event=entity.updated            actor=did:arc:demo:researcher/c3360492
  seq=5  event=entity.updated            actor=did:arc:demo:researcher/c3360492

chain valid? True  (verified through seq=5)


## 5. Member identity — `did` and `handle` are first-class fields

Earlier versions of `arcteam` treated the `Entity.id` URI as the only identity a
registry record carried, and bound a DID to a member by stuffing a `"did:..."`
string onto the free-form `capabilities` list. That's gone. `did` and `handle`
are now real, **required** `Entity` fields — every member is DID-keyed from
construction, and `arctrust` remains the source of truth for the cryptography
(`arcteam` never mints an identity on its own; it only stores the binding the
caller supplies). (See `arctrust/01-identity-did.ipynb` for the DID model
itself; we don't re-derive it here.)

**Registry handle vs. cryptographic identity, still distinct concepts:**

```
agent://planner   (Entity.id, URI address)
handle="planner"  (Entity.handle, @mention / display name)
did="did:arc:demo:planner/9b43ee77"   (Entity.did, verifiable identity)
```

`planner` was already registered with a real `did`/`handle` in section 4 (our
`_entity()` helper minted the `AgentIdentity` and stashed it in
`_entity_identities`). Below we pull that same identity back out to demonstrate
end-to-end signing and verification against the registry record — no second,
mismatched identity generated.

In [16]:
# Retrieve the SAME identity that minted planner's did/handle in section 4 —
# not a fresh one, which would not match the registered `did`.
planner_identity = _entity_identities["agent://planner"]
print("planner DID:        ", planner_identity.did)
print("matches registered: ", planner_identity.did == planner.did)
print("can sign?           ", planner_identity.can_sign)

planner DID:         did:arc:demo:planner/081db836
matches registered:  True
can sign?            True


Confirm the registry record already carries the binding — no side-channel `capabilities` hack needed.

In [17]:
# did/handle are already on the record — read them straight off.
planner_now = await registry.get("agent://planner")
assert planner_now is not None
print("planner.did:   ", planner_now.did)
print("planner.handle:", planner_now.handle)
assert planner_now.did == planner_identity.did

planner.did:    did:arc:demo:planner/081db836
planner.handle: planner


With `did` on the record itself, any caller holding the registry handle (`agent://planner`) can read the DID directly and verify a signature — no parsing a `capabilities` tag. This is how `arcteam` enables signed inter-agent messaging without baking signature logic into the registry itself: the registry stores the binding; `arctrust` provides verification.

In [18]:
# Demonstrate the verification path: caller has the handle, reads `did` off
# the record, verifies a signature with arctrust primitives.
from arctrust import sign as raw_sign, verify as raw_verify

# Planner signs a payload (e.g., a task assignment).
assert planner_identity._signing_key is not None
payload = b"assign:task-001:agent://researcher"
sig = raw_sign(payload, planner_identity._signing_key.encode())

# Verifier path: pull the registry record, read `did` straight off the field.
record = await registry.get("agent://planner")
assert record is not None
print("resolved DID:", record.did)

# Verify using the cached public key (in production you'd resolve via a trust store).
ok = raw_verify(payload, sig, planner_identity.public_key)
print("signature valid:", ok)
assert ok

resolved DID: did:arc:demo:planner/081db836
signature valid: True


**Where the line is.** `arcteam` does not re-implement signing, parent/child derivation, or the policy pipeline. Those primitives live in `arctrust` and are reused as-is — the registry is purely the place where team-scoped *handles* are kept. Notebooks 02 (channels) and 03 (messaging) show how signed messages travel between members; this notebook only sets up the membership graph.

## 6. Forming a team end-to-end — minimal example

Pulling everything together: a fresh team comes up in three steps:

1. **Choose a config and a backend.** `TeamConfig` for deployment knobs, `MemoryBackend` (tests) or `NatsBackend` (production, JetStream-backed) for storage.
2. **Stand up the audit logger.** `AuditLogger` with an Ed25519 signer — chains every team mutation.
3. **Register members.** `EntityRegistry.register(...)` for each `Entity`.

Below is the smallest end-to-end formation. We wrap it in a function so the same code can be reused later in tests.

In [19]:
async def form_team(
    name: str, members: list[Entity]
) -> tuple[EntityRegistry, AuditLogger, MemoryBackend]:
    """Stand up a fresh in-process team and register members."""
    backend = MemoryBackend()
    audit = AuditLogger(
        backend, signer=InProcessSigner(hashlib.sha256(f"{name}-key".encode()).digest(), ED25519)
    )
    await audit.initialize()
    registry = EntityRegistry(backend, audit)
    for member in members:
        await registry.register(member)
    return registry, audit, backend

Use the helper to form a small "research squad" team.

In [20]:
squad = [
    _entity(
        id="agent://lead",
        name="Lead",
        type=EntityType.AGENT,
        roles=["coordinator"],
        capabilities=["task.assign"],
    ),
    _entity(
        id="agent://scout",
        name="Scout",
        type=EntityType.AGENT,
        roles=["worker", "research"],
        capabilities=["web.search"],
    ),
    _entity(
        id="agent://writer",
        name="Writer",
        type=EntityType.AGENT,
        roles=["worker", "writing"],
        capabilities=["doc.draft"],
    ),
    _entity(id="user://owner", name="Owner", type=EntityType.USER, roles=["operator"]),
]

squad_registry, squad_audit, squad_backend = await form_team("research-squad", squad)
roster = await squad_registry.list_entities()
print(f"research-squad has {len(roster)} members:")
for m in roster:
    print(f"  {m.id:<24}  type={m.type.value:<6}  roles={m.roles}")

research-squad has 4 members:
  agent://lead              type=agent   roles=['coordinator']
  user://owner              type=user    roles=['operator']
  agent://scout             type=agent   roles=['worker', 'research']
  agent://writer            type=agent   roles=['worker', 'writing']


Sanity-check the audit chain from formation. Every member registered = one `entity.registered` event = one Ed25519-signed record.

In [21]:
formation_records = await squad_backend.read_stream("audit", "audit", after_seq=0)
print(f"Audit records during formation: {len(formation_records)}")
for r in formation_records:
    print(f"  seq={r['audit_seq']}  {r['event_type']}  actor={r['actor_id']}")
ok, last = await squad_audit.verify_chain()
print(f"chain valid? {ok}  (through seq={last})")
assert ok and last == len(squad)

Audit records during formation: 4
  seq=1  entity.registered  actor=did:arc:demo:lead/03682c1d
  seq=2  entity.registered  actor=did:arc:demo:scout/14eb66de
  seq=3  entity.registered  actor=did:arc:demo:writer/93cc25b5
  seq=4  entity.registered  actor=did:arc:demo:owner/971fc1da
chain valid? True  (through seq=4)


**Role-based addressing in action.** Once members are registered with `roles`, you can address subsets of the team by role rather than by individual ID. This is the foundation `arcteam`'s messaging layer (notebook `03`) builds `role://` URIs on top of.

In [22]:
workers = await squad_registry.by_role("worker")
print("Workers:")
for w in workers:
    print(f"  {w.id}  capabilities={w.capabilities}")

coordinators = await squad_registry.by_role("coordinator")
print("Coordinators:", [c.id for c in coordinators])

# Capability-based filtering is *not* a registry primitive — list and filter in code.
researchers = [e for e in roster if "research" in e.roles]
print("With 'research' role:", [r.id for r in researchers])

Workers:
  agent://scout  capabilities=['web.search']
  agent://writer  capabilities=['doc.draft']
Coordinators: ['agent://lead']
With 'research' role: ['agent://scout']


## 7. Member roles and metadata — what's representable

`Entity.roles` and `Entity.capabilities` are both `list[str]` — string-tag bags. That's a deliberate "boring" choice. Arc deployments differ wildly in their org structures (federal vs. enterprise vs. personal), so `arcteam` deliberately doesn't bake in a fixed role taxonomy. You choose.

That said, *conventions* matter — here's the convention that emerges across Arc components:

| Tag form | Example | Used for |
|---|---|---|
| **Functional roles** | `worker`, `coordinator`, `reviewer` | Role-based addressing (`role://worker`) |
| **Domain roles** | `research`, `writing`, `qa` | Subset filtering, capability discovery |
| **Operator roles** | `operator`, `owner`, `admin` | Authority gates (combine with policy in arctrust) |
| **Capability tags** | `web.search`, `doc.draft`, `sql` | Tool-level capability advertisement |
| **Identity bindings** | `Entity.did` (a first-class field, not a tag) | DID lookup (see §5) |

Pick a convention and stick to it across a deployment. The registry won't enforce one, but downstream addressing and policy lookups will assume it.

In [23]:
# Showcase the difference between role and capability tags.
multi_role_agent = _entity(
    id="agent://swiss-army",
    name="Swiss Army",
    type=EntityType.AGENT,
    roles=["worker", "research", "writing", "reviewer"],
    capabilities=["web.search", "doc.draft", "doc.review", "sql"],
)
await squad_registry.register(multi_role_agent)

# Same entity surfaces in multiple role queries.
for role in ("worker", "research", "writing", "reviewer"):
    matches = await squad_registry.by_role(role)
    ids = [m.id for m in matches]
    print(f"role '{role}':  {ids}")

role 'worker':  ['agent://scout', 'agent://swiss-army', 'agent://writer']
role 'research':  ['agent://scout', 'agent://swiss-army']
role 'writing':  ['agent://swiss-army', 'agent://writer']
role 'reviewer':  ['agent://swiss-army']


Capabilities aren't a registry-level filter. They're free-form tags that other layers (the agent's tool registry, a discovery service) interpret.

In [24]:
# Capability discovery — done in caller code, not by the registry.
def find_capable(entities: list[Entity], capability: str) -> list[Entity]:
    return [e for e in entities if capability in e.capabilities]


all_members = await squad_registry.list_entities()
sql_capable = find_capable(all_members, "sql")
print("SQL-capable agents:", [e.id for e in sql_capable])

draft_capable = find_capable(all_members, "doc.draft")
print("Drafting-capable agents:", [e.id for e in draft_capable])

SQL-capable agents: ['agent://swiss-army']
Drafting-capable agents: ['agent://swiss-army', 'agent://writer']


**`workspace_path` (SPEC-019).** Agents have a workspace directory on disk — populated either at registration time (`arc team register --workspace ...`) or backfilled with `arc team backfill-workspaces` for older records. Users have no workspace, so `workspace_path = None`. This is enforced by convention; the registry will accept any value.

In [25]:
# Show the workspace_path field — agent has one, user does not.
ws_records = [(e.id, e.type.value, e.workspace_path) for e in all_members]
for eid, etype, wp in ws_records:
    print(f"  {eid:<24}  type={etype:<6}  workspace={wp!r}")

  agent://lead              type=agent   workspace=None
  user://owner              type=user    workspace=None
  agent://scout             type=agent   workspace=None
  agent://swiss-army        type=agent   workspace=None
  agent://writer            type=agent   workspace=None


## 8. Lifecycle — formation → operation → dissolution

Teams aren't immortal. They're formed for a purpose, operate, and dissolve when the purpose is complete (or the SCIF audit window closes). `arcteam` tracks this lifecycle through role-tag conventions (not `Entity.status`, which today is fixed at `active` — see section 6) and the audit chain — there's no separate "team object" with a state machine; the team's lifecycle *is* the sequence of registry mutations.

| Phase | What happens | Audit signal |
|---|---|---|
| **Formation** | Members registered; roles assigned | One `entity.registered` per member |
| **Operation** | Role updates, workspace backfills | `entity.updated` |
| **Dissolution** | Members marked via a `"dissolved"` role tag (soft-remove); audit sealed | Trailing `entity.updated` events |

Below we walk a tiny team through all three phases and inspect the resulting audit timeline.

In [26]:
# --- Phase 1: Formation
sprint = [
    _entity(id="agent://owner", name="Owner", type=EntityType.AGENT, roles=["coordinator"]),
    _entity(id="agent://hand-1", name="Hand-1", type=EntityType.AGENT, roles=["worker"]),
    _entity(id="agent://hand-2", name="Hand-2", type=EntityType.AGENT, roles=["worker"]),
]
sprint_reg, sprint_audit, sprint_backend = await form_team("sprint", sprint)

formation_count = len(await sprint_backend.read_stream("audit", "audit", after_seq=0))
print(f"after formation: {formation_count} audit records")

after formation: 3 audit records


In [27]:
# --- Phase 2: Operation
# A worker briefly goes offline (role-tag toggle — `status` can't represent
# this, see section 6), comes back, then both workers backfill workspace paths.
await _mutate_roles(sprint_reg, "agent://hand-1", ["offline"])
await _mutate_roles(sprint_reg, "agent://hand-1", ["worker"])

sprint_workspaces = sandbox_root / "sprint-workspaces"
for worker_id in ("agent://hand-1", "agent://hand-2"):
    rec = await sprint_reg.get(worker_id)
    assert rec is not None
    rec.workspace_path = str(sprint_workspaces / worker_id.split("://")[-1])
    await sprint_reg.update(rec)

operation_count = len(await sprint_backend.read_stream("audit", "audit", after_seq=0))
print(f"after operation: {operation_count} audit records")

after operation: 7 audit records


In [28]:
# --- Phase 3: Dissolution
# Mark everyone dissolved via a role tag. Records remain for audit; no physical deletes.
for member in await sprint_reg.list_entities():
    await _mutate_roles(sprint_reg, member.id, ["dissolved"])

dissolution_records = await sprint_backend.read_stream("audit", "audit", after_seq=0)
print(f"after dissolution: {len(dissolution_records)} audit records total")
print()
print("Final timeline (last 8 events):")
for r in dissolution_records[-8:]:
    print(f"  seq={r['audit_seq']:>2}  {r['event_type']:<24}  {r['actor_id']}")

# Audit chain still verifies after every transition.
ok, last = await sprint_audit.verify_chain()
print()
print(f"audit chain valid? {ok}  (verified through seq={last})")
assert ok

after dissolution: 10 audit records total

Final timeline (last 8 events):
  seq= 3  entity.registered         did:arc:demo:hand-2/d0abc4f7
  seq= 4  entity.updated            did:arc:demo:hand-1/8998415f
  seq= 5  entity.updated            did:arc:demo:hand-1/8998415f
  seq= 6  entity.updated            did:arc:demo:hand-1/8998415f
  seq= 7  entity.updated            did:arc:demo:hand-2/d0abc4f7
  seq= 8  entity.updated            did:arc:demo:hand-1/8998415f
  seq= 9  entity.updated            did:arc:demo:hand-2/d0abc4f7
  seq=10  entity.updated            did:arc:demo:owner/ce436c66

audit chain valid? True  (verified through seq=10)


A few practical notes about lifecycle:

- **Soft-remove > hard delete.** The `"dissolved"` role tag keeps the record visible in `list_entities()` (unlike `status`, which today is fixed at `active` — see section 6). Filtering it out at the application layer is fine; rewriting history isn't.
- **The audit chain is your timeline.** `read_stream("audit", "audit", ...)` is the canonical replay surface. Notebook `04-team-persistence` covers durable backends and recovery.
- **Cross-team isolation.** Each team can run on its own `MemoryBackend` / `NatsBackend`. There's no global registry — that's by design (no singleton bottlenecks; teams are independent).

In [29]:
# Cross-team isolation: squad_registry and sprint_reg never see each other.
squad_ids = {e.id for e in await squad_registry.list_entities()}
sprint_ids = {e.id for e in await sprint_reg.list_entities()}
print("squad ids:  ", sorted(squad_ids))
print("sprint ids: ", sorted(sprint_ids))
print("intersection:", squad_ids & sprint_ids)
assert not (squad_ids & sprint_ids), "teams must be isolated"

squad ids:   ['agent://lead', 'agent://scout', 'agent://swiss-army', 'agent://writer', 'user://owner']
sprint ids:  ['agent://hand-1', 'agent://hand-2', 'agent://owner']
intersection: set()


### Cleanup

Wipe the temp directory we created for `TeamConfig.root` so this notebook is leave-no-trace.

In [30]:
import shutil

if sandbox_root.exists():
    shutil.rmtree(sandbox_root)
    print("Removed:", sandbox_root)

Removed: /var/folders/tc/283s9y5x2ws8rtq0c9skdn700000gn/T/arc_team_demo_1db6deji


## 9. Summary

**What you saw in this notebook:**

- Why `arcteam` exists — the multi-agent layer that orchestrates `ArcAgent` instances without duplicating agent internals.
- `TeamConfig` carries deployment knobs (`root`, `max_body_bytes`, `default_poll_limit`) and nothing else — the audit signer is supplied separately to `AuditLogger`, not read from `TeamConfig`; team identity and roster live elsewhere.
- `Entity` is the unit of membership: typed URI (`agent://`/`user://`), a required `did` + `handle`, name, type, roles, capabilities, status, optional workspace path.
- `EntityRegistry` registers, looks up, lists, filters by role, and updates records — every mutation emits a chained audit event.
- `did`/`handle` are required `Entity` fields (not a `capabilities`-list hack) — DIDs from `arctrust` bind the registry record to a verifiable cryptographic identity; `arcteam` keeps the binding, `arctrust` does the cryptography.
- A team comes up in three steps: pick `TeamConfig` + backend, stand up `AuditLogger`, register members.
- Roles and capabilities are deliberate string-tag bags — convention over schema.
- `Entity.status` is typed `EntityStatus`, but today the enum defines only `active` — there is no `update_status()` and no other legal status value. Assigning a non-enum string succeeds silently in memory (no `validate_assignment`) but breaks the next validated read. Lifecycle state (offline, retired, dissolved) is represented via **role tags** instead, mutated through `update()`.
- Lifecycle is *implicit* in the audit chain: formation → operation → dissolution is just a sequence of registry mutations, soft-removed via a role-tag `update()` (e.g. adding `"dissolved"`) rather than physical deletes.
- Cross-team isolation is a property of the backend, not a registry feature — no singleton, no shared state.

**Public API surface covered:**

- `arcteam.TeamConfig` — defaults, overrides, Pydantic round-trip
- `arcteam.Entity` and `arcteam.EntityType` (`AGENT` / `USER`)
- `arcteam.EntityRegistry` — `register`, `get`, `list_entities`, `by_role`, `update`
- `arcteam.MemoryBackend` (used as `StorageBackend`)
- `arcteam.AuditLogger` — initialize, `verify_chain`
- Cross-references: `arctrust.AgentIdentity`, `arctrust.sign`, `arctrust.verify` for DID-bound members

**Where this fits in the four pillars:**

- **Identity** — every team member maps to a DID via `arctrust` (covered in `arctrust/01-identity-did`).
- **Sign** — signed inter-agent messages use `arctrust.sign` / `arctrust.verify`; `arcteam` stores the binding only.
- **Authorize** — team-scoped policy decisions still go through `arctrust.policy` (`PolicyPipeline`); `arcteam` doesn't ship its own policy engine.
- **Audit** — every registry mutation lands in the Ed25519-signed audit stream. Notebook `04-team-persistence` covers durability and recovery.

**Next:**

- `arcteam/02-task-distribution.ipynb` — `Channel`, `MsgType`, `Priority`, and how tasks fan out across the roster.
- `arcteam/03-messaging-channels.ipynb` — `MessagingService` and the `Message` envelope; signed inter-agent communication.
- `arcteam/04-team-persistence.ipynb` — `TeamMemoryService`, the durable `NatsBackend` (JetStream), team recovery after a crash.